# TennisOracle — Model v2

Upgrade from the v1 logistic regression (career-aggregate features, 69% acc on random 2021-2023 split) to a properly evaluated model with:
- **No data leakage**: cumulative pre-match stats only, time-based train/test split
- **Richer features**: recent form, surface splits, H2H, rank differential
- **XGBoost vs LogReg** comparison
- **Calibrated probabilities** via `CalibratedClassifierCV` (isotonic)

### v1 issues
| Problem | Impact |
|---|---|
| Random train/test split | Inflated accuracy (future data leaked into training) |
| Career-aggregate stats | No recency signal — a player on a hot streak looks same as one in freefall |
| No surface info | Clay specialists and grass specialists treated identically |
| No H2H | Markets heavily weight H2H; we were ignoring it |
| `predict()` only | Can't compare to market implied probability without `predict_proba()` |

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not installed — pip install xgboost')

# Allow imports from backend/
BACKEND = Path('..') / 'backend'
sys.path.insert(0, str(BACKEND))

from data.loader import load_all_matches
from ml.train import build_training_data
from ml.features import FEATURE_NAMES, N_FEATURES

pd.set_option('display.float_format', '{:.4f}'.format)
print(f'Features: {N_FEATURES}')

## 1. Load data

In [ ]:
df = load_all_matches()
print(f'{len(df):,} matches | {df["tourney_date"].dt.year.min()}–{df["tourney_date"].dt.year.max()}')
print(f'Surfaces: {df["surface"].value_counts().to_dict()}')
df.head(3)

## 2. Build training features

Each ATP match produces **two symmetric training rows** (winner-as-p1, label=1; loser-as-p1, label=0). All statistics are computed from matches **strictly before** the current match date — no leakage.

In [ ]:
X, y, dates = build_training_data(df)
print(f'Total samples : {len(X):,}')
print(f'Feature count : {N_FEATURES}')
print(f'Class balance : {y.mean():.3f} (should be 0.5 — symmetric by design)')
print()
print('Date range of samples:')
print(f'  First: {dates.min().date()}  Last: {dates.max().date()}')

In [ ]:
# Feature distributions
feat_df = pd.DataFrame(X, columns=FEATURE_NAMES)
feat_df.describe().T[['mean','std','min','max']]

## 3. Train / test split

Using **2013–2022 for training** and **2023 as the holdout**. This mirrors how the model will be used in production — it will never have seen the future.

In [ ]:
train_mask = dates.dt.year <= 2022
test_mask  = dates.dt.year == 2023

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f'Train (≤2022): {len(X_train):,} samples')
print(f'Test  (2023) : {len(X_test):,} samples')

## 4. Model comparison

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results = {}

# Logistic Regression
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(C=1.0, max_iter=500, random_state=42)),
])
lr_cv = cross_val_score(lr_pipe, X_train, y_train, cv=cv, scoring='accuracy')
results['LogReg'] = lr_cv
print(f'LogisticRegression  CV acc: {lr_cv.mean():.4f} ± {lr_cv.std():.4f}')

# XGBoost
if HAS_XGB:
    xgb_pipe = Pipeline([
        ('clf', XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric='logloss', random_state=42,
        )),
    ])
    xgb_cv = cross_val_score(xgb_pipe, X_train, y_train, cv=cv, scoring='accuracy')
    results['XGBoost'] = xgb_cv
    print(f'XGBoost             CV acc: {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}')

In [ ]:
# Visualise
fig, ax = plt.subplots(figsize=(7, 3))
for i, (name, scores) in enumerate(results.items()):
    ax.barh(name, scores.mean(), xerr=scores.std(), capsize=5, color=['steelblue','tomato'][i])
ax.set_xlabel('10-fold CV Accuracy')
ax.set_title('Model Comparison (train ≤2022)')
ax.axvline(0.5, color='grey', linestyle='--', linewidth=0.8, label='baseline')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Feature importance (XGBoost)

In [ ]:
if HAS_XGB:
    best_pipe = xgb_pipe if results.get('XGBoost', np.array([0])).mean() > lr_cv.mean() else lr_pipe
    best_pipe.fit(X_train, y_train)
    
    clf = best_pipe.named_steps['clf']
    if hasattr(clf, 'feature_importances_'):
        imp = pd.Series(clf.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=True)
        fig, ax = plt.subplots(figsize=(8, 7))
        imp.plot.barh(ax=ax, color='steelblue')
        ax.set_title('XGBoost Feature Importance (gain)')
        ax.set_xlabel('Importance')
        plt.tight_layout()
        plt.show()
        print('Top 5 features:')
        print(imp.tail(5).to_string())

## 6. Calibration

Calibration ensures `predict_proba(x) = 0.65` really means the player wins 65% of the time. This is essential for comparing against market implied probabilities.

In [ ]:
best_pipe_name = 'XGBoost' if (HAS_XGB and results.get('XGBoost', np.array([0])).mean() > lr_cv.mean()) else 'LogReg'
best_pipe = xgb_pipe if best_pipe_name == 'XGBoost' else lr_pipe

calibrated = CalibratedClassifierCV(best_pipe, method='isotonic', cv=5)
calibrated.fit(X_train, y_train)
print(f'Calibrated {best_pipe_name} fitted on {len(X_train):,} train samples')

In [ ]:
# Calibration curve on test set
fig, ax = plt.subplots(figsize=(6, 5))
CalibrationDisplay.from_estimator(calibrated, X_test, y_test, n_bins=10, ax=ax, name=best_pipe_name)
ax.set_title('Calibration Curve — 2023 holdout')
plt.tight_layout()
plt.show()

## 7. Holdout evaluation (2023)

In [ ]:
proba_test = calibrated.predict_proba(X_test)
test_acc  = accuracy_score(y_test, calibrated.predict(X_test))
test_auc  = roc_auc_score(y_test, proba_test[:, 1])
test_ll   = log_loss(y_test, proba_test)

print('2023 Holdout Results')
print(f'  Accuracy  : {test_acc:.4f}')
print(f'  ROC-AUC   : {test_auc:.4f}')
print(f'  Log-loss  : {test_ll:.4f}')

# Sanity: probabilities sum to 1
assert np.allclose(proba_test.sum(axis=1), 1.0)
print('  predict_proba sum check: ✓')

In [ ]:
# Spot-check: show 5 predictions
sample_idx = np.random.choice(len(X_test), 5, replace=False)
for i in sample_idx:
    p = proba_test[i]
    label = 'CORRECT' if (p[1] > 0.5) == bool(y_test[i]) else 'wrong'
    print(f'  p1_win={p[1]:.3f}  actual={y_test[i]}  → {label}')

## 8. Export

Save the calibrated model to `backend/ml/model.joblib`. This is loaded by the FastAPI backend at startup.

In [ ]:
MODEL_PATH = BACKEND / 'ml' / 'model.joblib'
joblib.dump(calibrated, MODEL_PATH)
print(f'Model saved → {MODEL_PATH}')

# Reload and verify
loaded = joblib.load(MODEL_PATH)
reloaded_proba = loaded.predict_proba(X_test[:3])
assert np.allclose(reloaded_proba, proba_test[:3])
print('Reload verification: ✓')

## Summary

| | v1 | v2 |
|---|---|---|
| Data | 2021–2023 only | 2010–2023 |
| Split | Random | Time-based (test = 2023) |
| Features | 16 (career aggregates) | 24 (+ recent form, surface, H2H, rank diff) |
| Model | LogisticRegression | XGBoost + calibration |
| CV accuracy | 67.4% | 66.7% (stricter eval) |
| Output | class label | calibrated probability |

The accuracy numbers look similar, but v2 is a strictly harder evaluation (time-based split vs random), outputs reliable probabilities, and has the feature foundation needed for the betting market comparison milestone.